# SupportSense — LLM Fine-Tuning for Banking Intent Classification

> End-to-end fine-tuning of Qwen2.5-7B-Instruct using Unsloth, LoRA/QLoRA, and the BANKING77 dataset.

## Project Overview

SupportSense is an intent classification system that fine-tunes an open-source Large Language Model to understand customer banking queries and map them to predefined support intents.

### Objective

Given a customer query such as:

> "I am still waiting on my card."

the fine-tuned model should predict:

```text
card_arrival
```

### Key Results & Metrics

* **Accuracy:** **91.67%** on 300 unseen test examples
* **Correct Predictions:** 275 / 300
* **Training Subset:** 1,000 examples
* **Test Subset:** 300 examples
* **Base Model:** Qwen2.5-7B-Instruct
* **Fine-Tuning Method:** LoRA / QLoRA
* **Framework:** Unsloth
* **Hardware GPU:** Google Colab Tesla T4


In [ ]:
!nvidia-smi

Sat Aug 15 14:07:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.7/75.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3

In [ ]:
import unsloth

print("Unsloth installed successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth installed successfully!


## 1. Dataset Preparation

We use the **BANKING77** dataset for customer-support intent classification.

The dataset contains 77 banking-related intents.

For this learning experiment:

- Training subset: 1,000 examples
- Evaluation subset: 300 examples

Each example is transformed from:

```text
text + label
```

into a conversational format:

```text
user      → customer query
assistant → intent name
```

Example:

```text
User:
I am still waiting on my card?

Assistant:
card_arrival
```


In [ ]:
from datasets import load_dataset

dataset = load_dataset("PolyAI/banking77")

print(dataset)

dataset_infos.json:   0%|          | 0.00/5.89k [00:00<?, ?B/s]

The repository for PolyAI/banking77 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/PolyAI/banking77.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


/root/.cache/huggingface/modules/datasets_modules/datasets/PolyAI--banking77/17ffc2ed47c2ed928bee64127ff1dbc97204cb974c2f980becae7c864007aed9/banking77.py:25: SyntaxWarning: invalid escape sequence '\~'
  author      = {I{\~{n}}igo Casanueva and Tadas Temcinas and Daniela Gerz and Matthew Henderson and Ivan Vulic},


Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})


In [ ]:
print(dataset["train"][0])
print(dataset["train"][1])
print(dataset["train"][2])

{'text': 'I am still waiting on my card?', 'label': 11}
{'text': "What can I do if my card still hasn't arrived after 2 weeks?", 'label': 11}
{'text': 'I have been waiting over a week. Is the card still coming?', 'label': 11}


In [ ]:
!pip install -q "datasets<4.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 12.5 MB/s eta 0:00:00


In [ ]:
print(dataset["train"].features["label"].names[11])

card_arrival


In [ ]:
labels = dataset["train"].features["label"].names

for i, label in enumerate(labels):
    print(f"{i}: {label}")

0: activate_my_card
1: age_limit
2: apple_pay_or_google_pay
3: atm_support
4: automatic_top_up
5: balance_not_updated_after_bank_transfer
6: balance_not_updated_after_cheque_or_cash_deposit
7: beneficiary_not_allowed
8: cancel_transfer
9: card_about_to_expire
10: card_acceptance
11: card_arrival
12: card_delivery_estimate
13: card_linking
14: card_not_working
15: card_payment_fee_charged
16: card_payment_not_recognised
17: card_payment_wrong_exchange_rate
18: card_swallowed
19: cash_withdrawal_charge
20: cash_withdrawal_not_recognised
21: change_pin
22: compromised_card
23: contactless_not_working
24: country_support
25: declined_card_payment
26: declined_cash_withdrawal
27: declined_transfer
28: direct_debit_payment_not_recognised
29: disposable_card_limits
30: edit_personal_details
31: exchange_charge
32: exchange_rate
33: exchange_via_app
34: extra_charge_on_statement
35: failed_transfer
36: fiat_currency_support
37: get_disposable_virtual_card
38: get_physical_card
39: getting_spar

In [ ]:
from collections import Counter

label_counts = Counter(dataset["train"]["label"])

for label_id, count in sorted(label_counts.items()):
    print(f"{label_id}: {labels[label_id]} → {count} examples")

0: activate_my_card → 159 examples
1: age_limit → 110 examples
2: apple_pay_or_google_pay → 126 examples
3: atm_support → 87 examples
4: automatic_top_up → 127 examples
5: balance_not_updated_after_bank_transfer → 171 examples
6: balance_not_updated_after_cheque_or_cash_deposit → 181 examples
7: beneficiary_not_allowed → 156 examples
8: cancel_transfer → 157 examples
9: card_about_to_expire → 129 examples
10: card_acceptance → 59 examples
11: card_arrival → 153 examples
12: card_delivery_estimate → 112 examples
13: card_linking → 139 examples
14: card_not_working → 112 examples
15: card_payment_fee_charged → 187 examples
16: card_payment_not_recognised → 168 examples
17: card_payment_wrong_exchange_rate → 167 examples
18: card_swallowed → 61 examples
19: cash_withdrawal_charge → 177 examples
20: cash_withdrawal_not_recognised → 160 examples
21: change_pin → 122 examples
22: compromised_card → 86 examples
23: contactless_not_working → 35 examples
24: country_support → 129 examples
25: d

In [ ]:
example = dataset["train"][0]

print("Original:")
print(example)

print("\nFine-tuning format:")

formatted_example = {
    "messages": [
        {
            "role": "user",
            "content": example["text"]
        },
        {
            "role": "assistant",
            "content": labels[example["label"]]
        }
    ]
}

print(formatted_example)

Original:
{'text': 'I am still waiting on my card?', 'label': 11}

Fine-tuning format:
{'messages': [{'role': 'user', 'content': 'I am still waiting on my card?'}, {'role': 'assistant', 'content': 'card_arrival'}]}


In [ ]:
train_small = dataset["train"].select(range(1000))
test_small = dataset["test"].select(range(300))

print("Training examples:", len(train_small))
print("Testing examples:", len(test_small))

Training examples: 1000
Testing examples: 300


In [ ]:
def format_example(example):
    return {
        "messages": [
            {
                "role": "user",
                "content": example["text"]
            },
            {
                "role": "assistant",
                "content": labels[example["label"]]
            }
        ]
    }


train_formatted = train_small.map(format_example)
test_formatted = test_small.map(format_example)

print(train_formatted)
print(test_formatted)

print(train_formatted[0]["messages"])

Dataset({
    features: ['text', 'label', 'messages'],
    num_rows: 1000
})
Dataset({
    features: ['text', 'label', 'messages'],
    num_rows: 300
})
[{'role': 'user', 'content': 'I am still waiting on my card?'}, {'role': 'assistant', 'content': 'card_arrival'}]


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

save_dir = "/content/drive/MyDrive/SupportSense-FineTuning/dataset"
os.makedirs(save_dir, exist_ok=True)

train_formatted.to_json(
    f"{save_dir}/train.jsonl",
    orient="records",
    lines=True
)

test_formatted.to_json(
    f"{save_dir}/test.jsonl",
    orient="records",
    lines=True
)

print("Dataset saved successfully!")
print(save_dir)

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Dataset saved successfully!
/content/drive/MyDrive/SupportSense-FineTuning/dataset


In [ ]:
import json

train_path = "/content/drive/MyDrive/SupportSense-FineTuning/dataset/train.jsonl"

with open(train_path, "r") as f:
    first_example = json.loads(f.readline())

print(json.dumps(first_example, indent=2))

{
  "text": "I am still waiting on my card?",
  "label": 11,
  "messages": [
    {
      "role": "user",
      "content": "I am still waiting on my card?"
    },
    {
      "role": "assistant",
      "content": "card_arrival"
    }
  ]
}


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## 3. Parameter-Efficient Fine-Tuning with LoRA

Instead of updating all parameters of Qwen2.5-7B, we use LoRA (Low-Rank Adaptation).

### Configuration

- Total parameters: 7.65B
- Trainable parameters: 40.37M
- Trainable percentage: 0.53%

The original model weights remain frozen while the LoRA adapters are trained.

```text
Qwen2.5-7B
    │
    ├── Base weights → Frozen
    │
    └── LoRA adapters → Trainable
```

This makes fine-tuning significantly more memory-efficient.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.8.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
model.print_trainable_parameters()

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [ ]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [ ]:
def apply_chat_template(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_ready = train_formatted.map(apply_chat_template)
test_ready = test_formatted.map(apply_chat_template)

print(train_ready[0]["text"])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
I am still waiting on my card?<|im_end|>
<|im_start|>assistant
card_arrival<|im_end|>



In [ ]:
ready_dir = "/content/drive/MyDrive/SupportSense-FineTuning/dataset"

train_ready.to_json(
    f"{ready_dir}/train_ready.jsonl",
    orient="records",
    lines=True
)

test_ready.to_json(
    f"{ready_dir}/test_ready.jsonl",
    orient="records",
    lines=True
)

print("Training-ready dataset saved!")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Training-ready dataset saved!


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ready,
    eval_dataset=test_ready,
    dataset_text_field="text",
    max_seq_length=2048,
    args=SFTConfig(
        output_dir="/content/drive/MyDrive/SupportSense-FineTuning/outputs",
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=50,
        save_total_limit=2,
        fp16=True,
        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/300 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
from transformers import TextStreamer

prompt = "I am still waiting on my card. What should I do?"

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

text_streamer = TextStreamer(tokenizer)

_ = model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=50,
    use_cache=True
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
I am still waiting on my card. What should I do?<|im_end|>
<|im_start|>assistant


Both `max_new_tokens` (=50) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


If you're waiting for a card and haven't received it yet, here are some steps you can take:

1. **Check Your Email or Account**: Sometimes, important information about your order or the status of your card can be sent to your email


In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 2 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
50,0.635236,0.629917
100,0.607270,0.578925
150,0.477474,0.570496
200,0.448795,0.569771
250,0.496293,0.559890


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SupportSense-FineTuning/outputs/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SupportSense-FineTuning/outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SupportSense-FineTuning/outputs/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SupportSense-FineTuning/outputs/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SupportSense-FineTuning/outputs/checkpoint-250/tokenizer_config.json.


## 5. Training Results

The model was fine-tuned for 2 epochs on a Tesla T4 GPU.

| Metric | Result |
|---|---:|
| Training examples | 1,000 |
| Epochs | 2 |
| Trainable parameters | 40.37M |
| Total parameters | 7.65B |
| Trainable percentage | 0.53% |
| Training time | ~12 min |
| Test examples | 300 |
| Correct predictions | 275 |
| Accuracy | **91.67%** |

### Evaluation

The fine-tuned model achieved **91.67% accuracy** on the 300-example unseen test set.

This demonstrates that the model learned the target intent-classification behavior rather than simply generating general conversational responses.


In [ ]:
prompt = "I am still waiting on my card. What should I do?"

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=20,
    do_sample=False
)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

Both `max_new_tokens` (=20) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
I am still waiting on my card. What should I do?<|im_end|>
<|im_start|>assistant
card_arrival<|im_end|>


In [ ]:
test_example = test_formatted[0]

print("Question:")
print(test_example["text"])

print("\nExpected intent:")
print(labels[test_example["label"]])

Question:
How do I locate my card?

Expected intent:
card_arrival


In [ ]:
prompt = test_example["text"]

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=10,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0][inputs.shape[1]:],
    skip_special_tokens=True
).strip()

print("Question:", prompt)
print("Expected:", labels[test_example["label"]])
print("Predicted:", response)

Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How do I locate my card?
Expected: card_arrival
Predicted: card_linking


In [ ]:
correct = 0
total = len(test_formatted)

for i in range(total):
    prompt = test_formatted[i]["text"]
    expected = labels[test_formatted[i]["label"]]

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=10,
        do_sample=False
    )

    prediction = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=True
    ).strip()

    if prediction == expected:
        correct += 1

accuracy = correct / total * 100

print(f"Correct: {correct}/{total}")
print(f"Accuracy: {accuracy:.2f}%")

Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Correct: 275/300
Accuracy: 91.67%


In [ ]:
wrong_predictions = []

for i in range(len(test_formatted)):
    prompt = test_formatted[i]["text"]
    expected = labels[test_formatted[i]["label"]]

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    input_length = inputs["input_ids"].shape[1]

    outputs = model.generate(
        **inputs,
        max_length=input_length + 10,
        do_sample=False
    )

    prediction = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()

    if prediction != expected:
        wrong_predictions.append({
            "index": i,
            "question": prompt,
            "expected": expected,
            "predicted": prediction
        })

print("Total wrong:", len(wrong_predictions))

for item in wrong_predictions[:10]:
    print("\nQuestion:", item["question"])
    print("Expected:", item["expected"])
    print("Predicted:", item["predicted"])

Total wrong: 25

Question: How do I locate my card?
Expected: card_arrival
Predicted: card_linking

Question: What currencies is an exchange rate calculated in?
Expected: exchange_rate
Predicted: fiat_currency_support

Question: Why am I being charged more ?
Expected: card_payment_wrong_exchange_rate
Predicted: extra_charge_on_statement

Question: How can I check the exchange rate applied to my transaction?
Expected: card_payment_wrong_exchange_rate
Predicted: exchange_rate

Question: There is an incoming payment into my account, but it is deactivated. Will they still be processed?
Expected: fiat_currency_support
Predicted: pending_cash_withdrawal

Question: I need my card now!
Expected: card_delivery_estimate
Predicted: card_arrival

Question: my card was not in the mail again can you advise?
Expected: card_delivery_estimate
Predicted: card_arrival

Question: i need my card quick
Expected: card_delivery_estimate
Predicted: card_linking

Question: How long will it take to get to me?
Ex

In [ ]:
import pandas as pd

errors_df = pd.DataFrame(wrong_predictions)

display(errors_df[[
    "index",
    "question",
    "expected",
    "predicted"
]])

,index,question,expected,predicted
0,0,How do I locate my card?,card_arrival,card_linking
1,98,What currencies is an exchange rate calculated...,exchange_rate,fiat_currency_support
2,138,Why am I being charged more ?,card_payment_wrong_exchange_rate,extra_charge_on_statement
3,156,How can I check the exchange rate applied to m...,card_payment_wrong_exchange_rate,exchange_rate
4,246,"There is an incoming payment into my account, ...",fiat_currency_support,pending_cash_withdrawal
5,280,I need my card now!,card_delivery_estimate,card_arrival
6,281,my card was not in the mail again can you advise?,card_delivery_estimate,card_arrival
7,282,i need my card quick,card_delivery_estimate,card_linking
8,283,How long will it take to get to me?,card_delivery_estimate,card_arrival
9,284,I'm just wondering when my card will get here.,card_delivery_estimate,card_arrival


In [ ]:
final_model_dir = "/content/drive/MyDrive/SupportSense-FineTuning/final_adapter"

model.save_pretrained(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

print("Final LoRA adapter saved successfully!")
print(final_model_dir)

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SupportSense-FineTuning/final_adapter/tokenizer_config.json.


Final LoRA adapter saved successfully!
/content/drive/MyDrive/SupportSense-FineTuning/final_adapter


In [ ]:
from unsloth import FastLanguageModel

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

print("Base model loaded successfully!")

==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Base model loaded successfully!


In [ ]:
import gc
import torch

del model
del trainer

gc.collect()
torch.cuda.empty_cache()

print(f"Free GPU memory: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")

Free GPU memory: 13.12 GB


In [ ]:
from peft import PeftModel

adapter_path = "/content/drive/MyDrive/SupportSense-FineTuning/final_adapter"

base_model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

print("LoRA adapter loaded successfully!")

LoRA adapter loaded successfully!


In [ ]:
prompt = "I am still waiting on my card. What should I do?"

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

inputs = base_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = base_model.generate(
    input_ids=inputs,
    max_new_tokens=10,
    do_sample=False
)

response = base_tokenizer.decode(
    outputs[0][inputs.shape[1]:],
    skip_special_tokens=True
).strip()

print("Question:", prompt)
print("Predicted:", response)

Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: I am still waiting on my card. What should I do?
Predicted: card_arrival
